In [ ]:
%load_ext autoreload
%reload_ext autoreload

%autoreload 2
%matplotlib inline
import os 
#os.environ['CUDA_VISIBLE_DEVICES']=''
#import math
import tensorflow
from tensorflow.compat.v1.keras.backend import get_session
tensorflow.compat.v1.disable_v2_behavior()
import kerasAC 
from kerasAC.generators.snp_generator import *
from kerasAC.tiledb_config import *
from scipy.special import softmax
from kerasAC.interpret.deepshap import * 
from kerasAC.interpret.profile_shap import * 
from kerasAC.vis import * 
from kerasAC.helpers.transform_bpnet_io import * 

In [ ]:
#reading in the complete mutation list
case_snps_af_filtered_1=pd.read_csv('')
controls_snps_af_filtered_1=pd.read_csv('')


In [ ]:
dict_var_Score_cases={}
dict_var_Score_controls={}

In [ ]:
import glob
import re
#from keras.models import load_model, model_from_json
#from keras.utils.generic_utils import get_custom_objects
from tensorflow.keras.models import load_model, model_from_json
from tensorflow.keras.utils import get_custom_objects
from kerasAC.metrics import * 
from kerasAC.custom_losses import * 
from kerasAC.metrics import * 
from kerasAC.custom_losses import * 
#from keras_genomics.layers.convolutional import RevCompConv1D
import tensorflow_probability as tfp
peak_files=glob.glob('filter_mutations/asd_affiltered_hg38_within_peaks_*.bed')
print(peak_files)
print(len(peak_files))

In [ ]:

for zz in peak_files:
    name=zz.split('_')[-1].split('.')[0]
    print(name)
    case_snps_in_peaks=pd.read_csv(
    'filtered_mutations/asd_affiltered_hg38_within_peaks_'+str(name)+'.bed',sep='\t',header=None)
    case_snps_in_peaks.columns=['Chr','Pos0','Pos','rsid']
    case_snps_in_peaks.shape
    control_snps_in_peaks=pd.read_csv(
    'filtered_mutations/asd_ssc_control_affiltered_hg38_within_peaks_'+str(name)+'.bed',sep='\t',header=None)
    control_snps_in_peaks.columns=['Chr','Pos0','Pos','rsid']
    snps_inpeaks=case_snps_af_filtered_1.merge(case_snps_in_peaks,on=['Chr','Pos0','Pos','rsid'])
    print(snps_inpeaks.shape)
    snps_inpeaks=snps_inpeaks.drop_duplicates()
    control_snps_inpeaks=controls_snps_af_filtered_1.merge(control_snps_in_peaks,on=['Chr','Pos0','Pos','rsid'])
    print(control_snps_inpeaks.shape)
    snps_inpeaks.to_csv("filtered_mutations/asd_final_0base_"+str(name)+".tsv",header=True,index=False,sep='\t')
    control_snps_inpeaks.to_csv("filtered_mutations/asd_control_final_0base_"+str(name)+".tsv",header=True,index=False,sep='\t')
    for zz1 in range(0,5):
        #load the model! 
        model_num=str(zz1)
        custom_objects={"recall":recall,
                    "sensitivity":recall,
                    "specificity":specificity,
                    "fpr":fpr,
                    "fnr":fnr,
                    "precision":precision,
                    "f1":f1,
                    "ambig_binary_crossentropy":ambig_binary_crossentropy,
                    "ambig_mean_absolute_error":ambig_mean_absolute_error,
                    "ambig_mean_squared_error":ambig_mean_squared_error,
                    "MultichannelMultinomialNLL":MultichannelMultinomialNLL}
        get_custom_objects().update(custom_objects)
        import json 
        model=model_from_json(open("< google drive >"+name+"."+str(model_num)+".arch",'r').read())
        model.load_weights("< google drive >"+name+"."+str(model_num)+".weights")
        #reference allele sequence generator 
        ref_gen=SNPGenerator(bed_path="filtered_mutations/asd_final_0base_"+name+".tsv",
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Ref",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)
        #alternate allele sequence generator 
        alt_gen=SNPGenerator(bed_path="filtered_mutations/asd_final_0base_"+name+".tsv",
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Alt",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)
        #get the reference allele predictions 
        count_preds={} 
        profile_preds={} 
        snp_to_seq={} 
        for i in range(0,len(ref_gen)):
            print(str(i))
            cur_x=ref_gen[i] 
            batch_rsids=cur_x[0] 
            batch_preds=model.predict(cur_x[1])
            batch_preds_profile=batch_preds[0]
            batch_preds_count=batch_preds[1] 
            for batch_index in range(len(batch_rsids)): 
                cur_rsid=batch_rsids[batch_index]
                snp_to_seq[cur_rsid]={} 
                snp_to_seq[cur_rsid]['ref']=cur_x[1][batch_index,:,:]
                cur_pred_profile=batch_preds_profile[batch_index,:,:]
                cur_pred_count=batch_preds_count[batch_index,:][0]
                count_preds[cur_rsid]={}
                count_preds[cur_rsid]['ref']=cur_pred_count
                profile_preds[cur_rsid]={}
                profile_preds[cur_rsid]['ref']=cur_pred_profile 
        #get the alternate allele predictions 
        for i in range(0,len(alt_gen)):
            print(str(i))
            cur_x=alt_gen[i] 
            batch_rsids=cur_x[0] 
            batch_preds=model.predict(cur_x[1])
            batch_preds_profile=batch_preds[0]
            batch_preds_count=batch_preds[1] 
            for batch_index in range(len(batch_rsids)): 
                cur_rsid=batch_rsids[batch_index]
                snp_to_seq[cur_rsid]['alt']=cur_x[1][batch_index,:,:]
                cur_pred_profile=batch_preds_profile[batch_index,:,:]
                cur_pred_count=batch_preds_count[batch_index,:][0]
                count_preds[cur_rsid]['alt']=cur_pred_count
                profile_preds[cur_rsid]['alt']=cur_pred_profile 
        count_preds_cases=count_preds
        profile_preds_cases=profile_preds
        snp_to_seq_cases=snp_to_seq
        log_odd_changes_sum_10=[]
        log_odd_changes_sum_20=[]
        log_odd_changes_sum_50=[]
        log_odd_changes_sum_100=[]
        log_odd_changes_sum_150=[]
        log_odd_changes_sum_200=[]
        log_odd_changes_sum_250=[]
        log_odd_changes_sum_300=[]
        log_odd_changes_sum_1000=[]
        for i in profile_preds_cases.keys():
            pred_prob_ref=softmax(profile_preds_cases[i]['ref'],axis=0)
            pred_prob_alt=softmax(profile_preds_cases[i]['alt'],axis=0)
            log_odd_changes_sum_1000.append(np.sum(np.log(pred_prob_ref)-np.log(pred_prob_alt)))
            log_odd_changes_sum_10.append(np.sum(np.log(pred_prob_ref[495:505])-np.log(pred_prob_alt[495:505])))
            log_odd_changes_sum_20.append(np.sum(np.log(pred_prob_ref[490:510])-np.log(pred_prob_alt[490:510])))
            log_odd_changes_sum_50.append(np.sum(np.log(pred_prob_ref[475:525])-np.log(pred_prob_alt[475:525])))
            log_odd_changes_sum_100.append(np.sum(np.log(pred_prob_ref[450:550])-np.log(pred_prob_alt[450:550])))
            log_odd_changes_sum_150.append(np.sum(np.log(pred_prob_ref[425:575])-np.log(pred_prob_alt[425:575])))
            log_odd_changes_sum_200.append(np.sum(np.log(pred_prob_ref[400:600])-np.log(pred_prob_alt[400:600])))
            log_odd_changes_sum_250.append(np.sum(np.log(pred_prob_ref[375:625])-np.log(pred_prob_alt[375:625])))
            log_odd_changes_sum_300.append(np.sum(np.log(pred_prob_ref[350:650])-np.log(pred_prob_alt[350:650]))) 
        local_count_changes_10=[]
        local_count_changes_20=[]
        local_count_changes_50=[]
        local_count_changes_100=[]
        local_count_changes_150=[]
        local_count_changes_200=[]
        local_count_changes_250=[]
        local_count_changes_300=[]
        local_count_changes_1000=[]
        for i in profile_preds_cases.keys():
            pred_prob_ref=softmax(profile_preds_cases[i]['ref'],axis=0)
            pred_prob_alt=softmax(profile_preds_cases[i]['alt'],axis=0)
            count_pred_ref=count_preds_cases[i]['ref']
            count_pred_alt=count_preds_cases[i]['alt']
            local_count_changes_1000.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref-np.exp(count_pred_alt)*pred_prob_alt))
            local_count_changes_10.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[495:505]-np.exp(count_pred_alt)*pred_prob_alt[495:505]))
            local_count_changes_20.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[490:510]-np.exp(count_pred_alt)*pred_prob_alt[490:510]))
            local_count_changes_50.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[475:525]-np.exp(count_pred_alt)*pred_prob_alt[475:525]))
            local_count_changes_100.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[450:550]-np.exp(count_pred_alt)*pred_prob_alt[450:550]))
            local_count_changes_150.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[425:575]-np.exp(count_pred_alt)*pred_prob_alt[425:575]))
            local_count_changes_200.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[400:600]-np.exp(count_pred_alt)*pred_prob_alt[400:600]))
            local_count_changes_250.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[375:625]-np.exp(count_pred_alt)*pred_prob_alt[375:625]))
            local_count_changes_300.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[350:650]-np.exp(count_pred_alt)*pred_prob_alt[350:650]))
        profile_preds_df=pd.DataFrame.from_dict(profile_preds_cases,orient='index')
        profile_preds_df['log_odd_changes_sum_10']=log_odd_changes_sum_10
        profile_preds_df['log_odd_changes_sum_20']=log_odd_changes_sum_20
        profile_preds_df['log_odd_changes_sum_50']=log_odd_changes_sum_50
        profile_preds_df['log_odd_changes_sum_100']=log_odd_changes_sum_100
        profile_preds_df['log_odd_changes_sum_150']=log_odd_changes_sum_150
        profile_preds_df['log_odd_changes_sum_200']=log_odd_changes_sum_200
        profile_preds_df['log_odd_changes_sum_250']=log_odd_changes_sum_250
        profile_preds_df['log_odd_changes_sum_300']=log_odd_changes_sum_300
        profile_preds_df['log_odd_changes_sum_1000']=log_odd_changes_sum_1000
        profile_preds_df['local_count_changes_10']=local_count_changes_10
        profile_preds_df['local_count_changes_20']=local_count_changes_20
        profile_preds_df['local_count_changes_50']=local_count_changes_50
        profile_preds_df['local_count_changes_100']=local_count_changes_100
        profile_preds_df['local_count_changes_150']=local_count_changes_150
        profile_preds_df['local_count_changes_200']=local_count_changes_200
        profile_preds_df['local_count_changes_250']=local_count_changes_250
        profile_preds_df['local_count_changes_300']=local_count_changes_300
        profile_preds_df['local_count_changes_1000']=local_count_changes_1000
        #reference allele sequence generator 
        ref_gen=SNPGenerator(bed_path='filtered_mutations/asd_control_final_0base_'+str(name)+'.tsv',
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Ref",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)
        #alternate allele sequence generator 
        alt_gen=SNPGenerator(bed_path='filtered_mutations/asd_control_final_0base_'+str(name)+'.tsv',
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Alt",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)
        #get the reference allele predictions 
        count_preds={} 
        profile_preds={} 
        snp_to_seq={} 
        for i in range(0,len(ref_gen)):
            print(str(i))
            cur_x=ref_gen[i] 
            batch_rsids=cur_x[0] 
            batch_preds=model.predict(cur_x[1])
            batch_preds_profile=batch_preds[0]
            batch_preds_count=batch_preds[1] 
            for batch_index in range(len(batch_rsids)): 
                cur_rsid=batch_rsids[batch_index]
                snp_to_seq[cur_rsid]={} 
                snp_to_seq[cur_rsid]['ref']=cur_x[1][batch_index,:,:]
                cur_pred_profile=batch_preds_profile[batch_index,:,:]
                cur_pred_count=batch_preds_count[batch_index,:][0]
                count_preds[cur_rsid]={}
                count_preds[cur_rsid]['ref']=cur_pred_count
                profile_preds[cur_rsid]={}
                profile_preds[cur_rsid]['ref']=cur_pred_profile 
        #get the alternate allele predictions 
        for i in range(0,len(alt_gen)):
            print(str(i))
            cur_x=alt_gen[i] 
            batch_rsids=cur_x[0] 
            batch_preds=model.predict(cur_x[1])
            batch_preds_profile=batch_preds[0]
            batch_preds_count=batch_preds[1] 
            for batch_index in range(len(batch_rsids)): 
                cur_rsid=batch_rsids[batch_index]
                snp_to_seq[cur_rsid]['alt']=cur_x[1][batch_index,:,:]
                cur_pred_profile=batch_preds_profile[batch_index,:,:]
                cur_pred_count=batch_preds_count[batch_index,:][0]
                count_preds[cur_rsid]['alt']=cur_pred_count
                profile_preds[cur_rsid]['alt']=cur_pred_profile 
        count_preds_controls=count_preds
        profile_preds_controls=profile_preds
        snp_to_seq_controls=snp_to_seq
        log_odd_changes_sum_10=[]
        log_odd_changes_sum_20=[]
        log_odd_changes_sum_50=[]
        log_odd_changes_sum_100=[]
        log_odd_changes_sum_150=[]
        log_odd_changes_sum_200=[]
        log_odd_changes_sum_250=[]
        log_odd_changes_sum_300=[]
        log_odd_changes_sum_1000=[]
        for i in profile_preds_controls.keys():
            pred_prob_ref=softmax(profile_preds_controls[i]['ref'],axis=0)
            pred_prob_alt=softmax(profile_preds_controls[i]['alt'],axis=0)
            log_odd_changes_sum_1000.append(np.sum(np.log(pred_prob_ref)-np.log(pred_prob_alt)))
            log_odd_changes_sum_10.append(np.sum(np.log(pred_prob_ref[495:505])-np.log(pred_prob_alt[495:505])))
            log_odd_changes_sum_20.append(np.sum(np.log(pred_prob_ref[490:510])-np.log(pred_prob_alt[490:510])))
            log_odd_changes_sum_50.append(np.sum(np.log(pred_prob_ref[475:525])-np.log(pred_prob_alt[475:525])))
            log_odd_changes_sum_100.append(np.sum(np.log(pred_prob_ref[450:550])-np.log(pred_prob_alt[450:550])))
            log_odd_changes_sum_150.append(np.sum(np.log(pred_prob_ref[425:575])-np.log(pred_prob_alt[425:575])))
            log_odd_changes_sum_200.append(np.sum(np.log(pred_prob_ref[400:600])-np.log(pred_prob_alt[400:600])))
            log_odd_changes_sum_250.append(np.sum(np.log(pred_prob_ref[375:625])-np.log(pred_prob_alt[375:625])))
            log_odd_changes_sum_300.append(np.sum(np.log(pred_prob_ref[350:650])-np.log(pred_prob_alt[350:650])))
        local_count_changes_10=[]
        local_count_changes_20=[]
        local_count_changes_50=[]
        local_count_changes_100=[]
        local_count_changes_150=[]
        local_count_changes_200=[]
        local_count_changes_250=[]
        local_count_changes_300=[]
        local_count_changes_1000=[]
        for i in profile_preds_controls.keys():
            pred_prob_ref=softmax(profile_preds_controls[i]['ref'],axis=0)
            pred_prob_alt=softmax(profile_preds_controls[i]['alt'],axis=0)
            count_pred_ref=count_preds_controls[i]['ref']
            count_pred_alt=count_preds_controls[i]['alt']
            local_count_changes_1000.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref-np.exp(count_pred_alt)*pred_prob_alt))
            local_count_changes_10.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[495:505]-np.exp(count_pred_alt)*pred_prob_alt[495:505]))
            local_count_changes_20.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[490:510]-np.exp(count_pred_alt)*pred_prob_alt[490:510]))
            local_count_changes_50.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[475:525]-np.exp(count_pred_alt)*pred_prob_alt[475:525]))
            local_count_changes_100.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[450:550]-np.exp(count_pred_alt)*pred_prob_alt[450:550]))
            local_count_changes_150.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[425:575]-np.exp(count_pred_alt)*pred_prob_alt[425:575]))
            local_count_changes_200.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[400:600]-np.exp(count_pred_alt)*pred_prob_alt[400:600]))
            local_count_changes_250.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[375:625]-np.exp(count_pred_alt)*pred_prob_alt[375:625]))
            local_count_changes_300.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[350:650]-np.exp(count_pred_alt)*pred_prob_alt[350:650]))
        profile_preds_df_controls=pd.DataFrame.from_dict(profile_preds_controls,orient='index')
        profile_preds_df_controls['log_odd_changes_sum_10']=log_odd_changes_sum_10
        profile_preds_df_controls['log_odd_changes_sum_20']=log_odd_changes_sum_20
        profile_preds_df_controls['log_odd_changes_sum_50']=log_odd_changes_sum_50
        profile_preds_df_controls['log_odd_changes_sum_100']=log_odd_changes_sum_100
        profile_preds_df_controls['log_odd_changes_sum_150']=log_odd_changes_sum_150
        profile_preds_df_controls['log_odd_changes_sum_200']=log_odd_changes_sum_200
        profile_preds_df_controls['log_odd_changes_sum_250']=log_odd_changes_sum_250
        profile_preds_df_controls['log_odd_changes_sum_300']=log_odd_changes_sum_300
        profile_preds_df_controls['log_odd_changes_sum_1000']=log_odd_changes_sum_1000
        profile_preds_df_controls['local_count_changes_10']=local_count_changes_10
        profile_preds_df_controls['local_count_changes_20']=local_count_changes_20
        profile_preds_df_controls['local_count_changes_50']=local_count_changes_50
        profile_preds_df_controls['local_count_changes_100']=local_count_changes_100
        profile_preds_df_controls['local_count_changes_150']=local_count_changes_150
        profile_preds_df_controls['local_count_changes_200']=local_count_changes_200
        profile_preds_df_controls['local_count_changes_250']=local_count_changes_250
        profile_preds_df_controls['local_count_changes_300']=local_count_changes_300
        profile_preds_df_controls['local_count_changes_1000']=local_count_changes_1000
        dict_var_Score_cases[name+'_'+str(model_num)]=profile_preds_df
        dict_var_Score_controls[name+'_'+str(model_num)]=profile_preds_df_controls
    

In [ ]:
dict_var_Score_cases_100_counts={}
dict_var_Score_controls_100_counts={}
dict_var_Score_cases_200_counts={}
dict_var_Score_controls_200_counts={}
dict_var_Score_cases_300_counts={}
dict_var_Score_controls_300_counts={}

dict_var_Score_cases_100_odds={}
dict_var_Score_controls_100_odds={}
dict_var_Score_cases_200_odds={}
dict_var_Score_controls_200_odds={}
dict_var_Score_cases_300_odds={}
dict_var_Score_controls_300_odds={}

In [ ]:
import tensorflow_probability as tfp
peak_files=glob.glob('filtered_mutations/asd_affiltered_hg38_within_peaks_*.bed')
for zz in peak_files:
    name=zz.split('_')[-1].split('.')[0]
    print(name)
    c0_fold0_cases=dict_var_Score_cases[name+'_0']
    c0_fold0_controls=dict_var_Score_controls[name+'_0']
    c0_fold1_cases=dict_var_Score_cases[name+'_1']
    c0_fold1_controls=dict_var_Score_controls[name+'_1']
    c0_fold2_cases=dict_var_Score_cases[name+'_2']
    c0_fold2_controls=dict_var_Score_controls[name+'_2']
    c0_fold3_cases=dict_var_Score_cases[name+'_3']
    c0_fold3_controls=dict_var_Score_controls[name+'_3']
    c0_fold4_cases=dict_var_Score_cases[name+'_4']
    c0_fold4_controls=dict_var_Score_controls[name+'_4']
    #
    del c0_fold0_cases['ref']
    del c0_fold0_cases['alt']
    del c0_fold0_controls['ref']
    del c0_fold0_controls['alt']
    #
    del c0_fold1_cases['ref']
    del c0_fold1_cases['alt']
    del c0_fold1_controls['ref']
    del c0_fold1_controls['alt']
    #
    del c0_fold2_cases['ref']
    del c0_fold2_cases['alt']
    del c0_fold2_controls['ref']
    del c0_fold2_controls['alt']
    #
    del c0_fold3_cases['ref']
    del c0_fold3_cases['alt']
    del c0_fold3_controls['ref']
    del c0_fold3_controls['alt']
    #
    del c0_fold4_cases['ref']
    del c0_fold4_cases['alt']
    del c0_fold4_controls['ref']
    del c0_fold4_controls['alt']
    #
    c0_fold0_cases.columns=['log_odd_changes_sum_10_1', 'log_odd_changes_sum_20_1',
       'log_odd_changes_sum_50_1', 'log_odd_changes_sum_100_1',
       'log_odd_changes_sum_150_1', 'log_odd_changes_sum_200_1',
       'log_odd_changes_sum_250_1', 'log_odd_changes_sum_300_1',
       'log_odd_changes_sum_1000_1', 
       'local_count_changes_10_1', 'local_count_changes_20_1',
       'local_count_changes_50_1', 'local_count_changes_100_1',
       'local_count_changes_150_1', 'local_count_changes_200_1',
       'local_count_changes_250_1', 'local_count_changes_300_1',
       'local_count_changes_1000_1']
    c0_fold1_cases.columns=['log_odd_changes_sum_10_2', 'log_odd_changes_sum_20_2',
       'log_odd_changes_sum_50_2', 'log_odd_changes_sum_100_2',
       'log_odd_changes_sum_150_2', 'log_odd_changes_sum_200_2',
       'log_odd_changes_sum_250_2', 'log_odd_changes_sum_300_2',
       'log_odd_changes_sum_1000_2', 
       'local_count_changes_10_2', 'local_count_changes_20_2',
       'local_count_changes_50_2', 'local_count_changes_100_2',
       'local_count_changes_150_2', 'local_count_changes_200_2',
       'local_count_changes_250_2', 'local_count_changes_300_2',
       'local_count_changes_1000_2']
    c0_fold2_cases.columns=['log_odd_changes_sum_10_3', 'log_odd_changes_sum_20_3',
       'log_odd_changes_sum_50_3', 'log_odd_changes_sum_100_3',
       'log_odd_changes_sum_150_3', 'log_odd_changes_sum_200_3',
       'log_odd_changes_sum_250_3', 'log_odd_changes_sum_300_3',
       'log_odd_changes_sum_1000_3', 
       'local_count_changes_10_3', 'local_count_changes_20_3',
       'local_count_changes_50_3', 'local_count_changes_100_3',
       'local_count_changes_150_3', 'local_count_changes_200_3',
       'local_count_changes_250_3', 'local_count_changes_300_3',
       'local_count_changes_1000_3']
    c0_fold3_cases.columns=['log_odd_changes_sum_10_4', 'log_odd_changes_sum_20_4',
       'log_odd_changes_sum_50_4', 'log_odd_changes_sum_100_4',
       'log_odd_changes_sum_150_4', 'log_odd_changes_sum_200_4',
       'log_odd_changes_sum_250_4', 'log_odd_changes_sum_300_4',
       'log_odd_changes_sum_1000_4', 
       'local_count_changes_10_4', 'local_count_changes_20_4',
       'local_count_changes_50_4', 'local_count_changes_100_4',
       'local_count_changes_150_4', 'local_count_changes_200_4',
       'local_count_changes_250_4', 'local_count_changes_300_4',
       'local_count_changes_1000_4']
    c0_fold4_cases.columns=['log_odd_changes_sum_10_5', 'log_odd_changes_sum_20_5',
       'log_odd_changes_sum_50_5', 'log_odd_changes_sum_100_5',
       'log_odd_changes_sum_150_5', 'log_odd_changes_sum_200_5',
       'log_odd_changes_sum_250_5', 'log_odd_changes_sum_300_5',
       'log_odd_changes_sum_1000_5', 
       'local_count_changes_10_5', 'local_count_changes_20_5',
       'local_count_changes_50_5', 'local_count_changes_100_5',
       'local_count_changes_150_5', 'local_count_changes_200_5',
       'local_count_changes_250_5', 'local_count_changes_300_5',
       'local_count_changes_1000_5']
    c0_fold0_controls.columns=['log_odd_changes_sum_10_1', 'log_odd_changes_sum_20_1',
       'log_odd_changes_sum_50_1', 'log_odd_changes_sum_100_1',
       'log_odd_changes_sum_150_1', 'log_odd_changes_sum_200_1',
       'log_odd_changes_sum_250_1', 'log_odd_changes_sum_300_1',
       'log_odd_changes_sum_1000_1', 
       'local_count_changes_10_1', 'local_count_changes_20_1',
       'local_count_changes_50_1', 'local_count_changes_100_1',
       'local_count_changes_150_1', 'local_count_changes_200_1',
       'local_count_changes_250_1', 'local_count_changes_300_1',
       'local_count_changes_1000_1']
    c0_fold1_controls.columns=['log_odd_changes_sum_10_2', 'log_odd_changes_sum_20_2',
       'log_odd_changes_sum_50_2', 'log_odd_changes_sum_100_2',
       'log_odd_changes_sum_150_2', 'log_odd_changes_sum_200_2',
       'log_odd_changes_sum_250_2', 'log_odd_changes_sum_300_2',
       'log_odd_changes_sum_1000_2', 
       'local_count_changes_10_2', 'local_count_changes_20_2',
       'local_count_changes_50_2', 'local_count_changes_100_2',
       'local_count_changes_150_2', 'local_count_changes_200_2',
       'local_count_changes_250_2', 'local_count_changes_300_2',
       'local_count_changes_1000_2']
    c0_fold2_controls.columns=['log_odd_changes_sum_10_3', 'log_odd_changes_sum_20_3',
       'log_odd_changes_sum_50_3', 'log_odd_changes_sum_100_3',
       'log_odd_changes_sum_150_3', 'log_odd_changes_sum_200_3',
       'log_odd_changes_sum_250_3', 'log_odd_changes_sum_300_3',
       'log_odd_changes_sum_1000_3', 
       'local_count_changes_10_3', 'local_count_changes_20_3',
       'local_count_changes_50_3', 'local_count_changes_100_3',
       'local_count_changes_150_3', 'local_count_changes_200_3',
       'local_count_changes_250_3', 'local_count_changes_300_3',
       'local_count_changes_1000_3']
    c0_fold3_controls.columns=['log_odd_changes_sum_10_4', 'log_odd_changes_sum_20_4',
       'log_odd_changes_sum_50_4', 'log_odd_changes_sum_100_4',
       'log_odd_changes_sum_150_4', 'log_odd_changes_sum_200_4',
       'log_odd_changes_sum_250_4', 'log_odd_changes_sum_300_4',
       'log_odd_changes_sum_1000_4', 
       'local_count_changes_10_4', 'local_count_changes_20_4',
       'local_count_changes_50_4', 'local_count_changes_100_4',
       'local_count_changes_150_4', 'local_count_changes_200_4',
       'local_count_changes_250_4', 'local_count_changes_300_4',
       'local_count_changes_1000_4']
    c0_fold4_controls.columns=['log_odd_changes_sum_10_5', 'log_odd_changes_sum_20_5',
       'log_odd_changes_sum_50_5', 'log_odd_changes_sum_100_5',
       'log_odd_changes_sum_150_5', 'log_odd_changes_sum_200_5',
       'log_odd_changes_sum_250_5', 'log_odd_changes_sum_300_5',
       'log_odd_changes_sum_1000_5', 
       'local_count_changes_10_5', 'local_count_changes_20_5',
       'local_count_changes_50_5', 'local_count_changes_100_5',
       'local_count_changes_150_5', 'local_count_changes_200_5',
       'local_count_changes_250_5', 'local_count_changes_300_5',
       'local_count_changes_1000_5']
    #
    c0_local_count100_cases=c0_fold0_cases[['local_count_changes_100_1']].merge(c0_fold1_cases[['local_count_changes_100_2']],left_index=True,right_index=True).\
    merge(c0_fold2_cases[['local_count_changes_100_3']],left_index=True,right_index=True).\
    merge(c0_fold3_cases[['local_count_changes_100_4']],left_index=True,right_index=True).\
    merge(c0_fold4_cases[['local_count_changes_100_5']],left_index=True,right_index=True)
    #xq
    c0_local_count200_cases=c0_fold0_cases[['local_count_changes_200_1']].merge(c0_fold1_cases[['local_count_changes_200_2']],left_index=True,right_index=True).\
    merge(c0_fold2_cases[['local_count_changes_200_3']],left_index=True,right_index=True).\
    merge(c0_fold3_cases[['local_count_changes_200_4']],left_index=True,right_index=True).\
    merge(c0_fold4_cases[['local_count_changes_200_5']],left_index=True,right_index=True)
    #
    c0_local_count300_cases=c0_fold0_cases[['local_count_changes_300_1']].merge(c0_fold1_cases[['local_count_changes_300_2']],left_index=True,right_index=True).\
    merge(c0_fold2_cases[['local_count_changes_300_3']],left_index=True,right_index=True).\
    merge(c0_fold3_cases[['local_count_changes_300_4']],left_index=True,right_index=True).\
    merge(c0_fold4_cases[['local_count_changes_300_5']],left_index=True,right_index=True)
    #
    c0_local_count100_controls=c0_fold0_controls[['local_count_changes_100_1']].merge(c0_fold1_controls[['local_count_changes_100_2']],left_index=True,right_index=True).\
    merge(c0_fold2_controls[['local_count_changes_100_3']],left_index=True,right_index=True).\
    merge(c0_fold3_controls[['local_count_changes_100_4']],left_index=True,right_index=True).\
    merge(c0_fold4_controls[['local_count_changes_100_5']],left_index=True,right_index=True)
    #
    c0_local_count200_controls=c0_fold0_controls[['local_count_changes_200_1']].merge(c0_fold1_controls[['local_count_changes_200_2']],left_index=True,right_index=True).\
    merge(c0_fold2_controls[['local_count_changes_200_3']],left_index=True,right_index=True).\
    merge(c0_fold3_controls[['local_count_changes_200_4']],left_index=True,right_index=True).\
    merge(c0_fold4_controls[['local_count_changes_200_5']],left_index=True,right_index=True)
    #
    c0_local_count300_controls=c0_fold0_controls[['local_count_changes_300_1']].merge(c0_fold1_controls[['local_count_changes_300_2']],left_index=True,right_index=True).\
    merge(c0_fold2_controls[['local_count_changes_300_3']],left_index=True,right_index=True).\
    merge(c0_fold3_controls[['local_count_changes_300_4']],left_index=True,right_index=True).\
    merge(c0_fold4_controls[['local_count_changes_300_5']],left_index=True,right_index=True)
    #
    c0_local_odd100_cases=c0_fold0_cases[['log_odd_changes_sum_100_1']].merge(c0_fold1_cases[['log_odd_changes_sum_100_2']],left_index=True,right_index=True).\
    merge(c0_fold2_cases[['log_odd_changes_sum_100_3']],left_index=True,right_index=True).\
    merge(c0_fold3_cases[['log_odd_changes_sum_100_4']],left_index=True,right_index=True).\
    merge(c0_fold4_cases[['log_odd_changes_sum_100_5']],left_index=True,right_index=True)
    #
    c0_local_odd200_cases=c0_fold0_cases[['log_odd_changes_sum_200_1']].merge(c0_fold1_cases[['log_odd_changes_sum_200_2']],left_index=True,right_index=True).\
    merge(c0_fold2_cases[['log_odd_changes_sum_200_3']],left_index=True,right_index=True).\
    merge(c0_fold3_cases[['log_odd_changes_sum_200_4']],left_index=True,right_index=True).\
    merge(c0_fold4_cases[['log_odd_changes_sum_200_5']],left_index=True,right_index=True)
    #
    c0_local_odd300_cases=c0_fold0_cases[['log_odd_changes_sum_300_1']].merge(c0_fold1_cases[['log_odd_changes_sum_300_2']],left_index=True,right_index=True).\
    merge(c0_fold2_cases[['log_odd_changes_sum_300_3']],left_index=True,right_index=True).\
    merge(c0_fold3_cases[['log_odd_changes_sum_300_4']],left_index=True,right_index=True).\
    merge(c0_fold4_cases[['log_odd_changes_sum_300_5']],left_index=True,right_index=True)
    #
    c0_local_odd100_controls=c0_fold0_controls[['log_odd_changes_sum_100_1']].merge(c0_fold1_controls[['log_odd_changes_sum_100_2']],left_index=True,right_index=True).\
    merge(c0_fold2_controls[['log_odd_changes_sum_100_3']],left_index=True,right_index=True).\
    merge(c0_fold3_controls[['log_odd_changes_sum_100_4']],left_index=True,right_index=True).\
    merge(c0_fold4_controls[['log_odd_changes_sum_100_5']],left_index=True,right_index=True)
    #
    c0_local_odd200_controls=c0_fold0_controls[['log_odd_changes_sum_200_1']].merge(c0_fold1_controls[['log_odd_changes_sum_200_2']],left_index=True,right_index=True).\
    merge(c0_fold2_controls[['log_odd_changes_sum_200_3']],left_index=True,right_index=True).\
    merge(c0_fold3_controls[['log_odd_changes_sum_200_4']],left_index=True,right_index=True).\
    merge(c0_fold4_controls[['log_odd_changes_sum_200_5']],left_index=True,right_index=True)
    #
    c0_local_odd300_controls=c0_fold0_controls[['log_odd_changes_sum_300_1']].merge(c0_fold1_controls[['log_odd_changes_sum_300_2']],left_index=True,right_index=True).\
    merge(c0_fold2_controls[['log_odd_changes_sum_300_3']],left_index=True,right_index=True).\
    merge(c0_fold3_controls[['log_odd_changes_sum_300_4']],left_index=True,right_index=True).\
    merge(c0_fold4_controls[['log_odd_changes_sum_300_5']],left_index=True,right_index=True)
    #
    c0_local_count100_cases['mean']=np.mean(c0_local_count100_cases,axis=1)
    c0_local_count100_controls['mean']=np.mean(c0_local_count100_controls,axis=1)
    c0_local_count200_cases['mean']=np.mean(c0_local_count200_cases,axis=1)
    c0_local_count200_controls['mean']=np.mean(c0_local_count200_controls,axis=1)
    c0_local_count300_cases['mean']=np.mean(c0_local_count300_cases,axis=1)
    c0_local_count300_controls['mean']=np.mean(c0_local_count300_controls,axis=1)
    #
    c0_local_odd100_cases['mean']=np.mean(c0_local_odd100_cases,axis=1)
    c0_local_odd100_controls['mean']=np.mean(c0_local_odd100_controls,axis=1)
    c0_local_odd200_cases['mean']=np.mean(c0_local_odd200_cases,axis=1)
    c0_local_odd200_controls['mean']=np.mean(c0_local_odd200_controls,axis=1)
    c0_local_odd300_cases['mean']=np.mean(c0_local_odd300_cases,axis=1)
    c0_local_odd300_controls['mean']=np.mean(c0_local_odd300_controls,axis=1)
    #
    dict_var_Score_cases_100_counts[name+'_cases_100_counts']=c0_local_count100_cases
    dict_var_Score_controls_100_counts[name+'_controls_100_counts']=c0_local_count100_controls
    dict_var_Score_cases_200_counts[name+'_cases_200_counts']=c0_local_count200_cases
    dict_var_Score_controls_200_counts[name+'_controls_200_counts']=c0_local_count200_controls
    dict_var_Score_cases_300_counts[name+'_cases_300_counts']=c0_local_count300_cases
    dict_var_Score_controls_300_counts[name+'_controls_300_counts']=c0_local_count300_controls
    #
    dict_var_Score_cases_100_odds[name+'_cases_100_odds']=c0_local_odd100_cases
    dict_var_Score_controls_100_odds[name+'_controls_100_odds']=c0_local_odd100_controls
    dict_var_Score_cases_200_odds[name+'_cases_200_odds']=c0_local_odd200_cases
    dict_var_Score_controls_200_odds[name+'_controls_200_odds']=c0_local_odd200_controls
    dict_var_Score_cases_300_odds[name+'_cases_300_odds']=c0_local_odd300_cases
    dict_var_Score_controls_300_odds[name+'_controls_300_odds']=c0_local_odd300_controls
    #


In [ ]:
prioritized_mutations_case=[]
prioritized_mutations_control=[]
for keyname in list(dict_var_Score_cases_100_odds.keys()):
  if keyname!='all_cases_100_odds':  
    try:
        print(keyname)
        colname='mean'
        control_name=keyname.replace('cases','controls')
        temp_df_cases=dict_var_Score_cases_100_odds[keyname]
        temp_df_controls=dict_var_Score_controls_100_odds[control_name]
        print(temp_df_cases[(temp_df_cases[colname]>20)].shape)
        print(temp_df_controls[(temp_df_controls[colname]>20)].shape)
        print(temp_df_cases[(temp_df_cases[colname]>20)].shape[0]/temp_df_controls[(temp_df_controls[colname]>20)].shape[0])
        #
        print(temp_df_cases[0:5])
        if temp_df_cases[(temp_df_cases[colname]>20)].shape[0]/temp_df_controls[(temp_df_controls[colname]>20)].shape[0] > 0:
            case_Df=temp_df_cases[(temp_df_cases[colname]>20)][['mean']]
            control_Df=temp_df_controls[(temp_df_controls[colname]>20)][['mean']]
            case_Df.columns=[keyname]
            control_Df.columns=[keyname]
            prioritized_mutations_case.append(case_Df)
            prioritized_mutations_control.append(control_Df)
        #break
        #
    except Exception as e :
        print(str(e))





In [ ]:
prioritized_mutations_case=[]
prioritized_mutations_control=[]

case_df=pd.DataFrame()
case_df=case_df.append(prioritized_mutations_case[0:1])

for i in range(1,len(prioritized_mutations_case)):
    tempdf=prioritized_mutations_case[i:i+1][0]
    #print(tempdf[0:5])
    case_df=case_df.merge(tempdf,left_index=True,right_index=True,how='outer')

case_df[0:5]
print(case_df.shape)

control_df=pd.DataFrame()
control_df=control_df.append(prioritized_mutations_control[0:1])

for i in range(1,len(prioritized_mutations_control)):
    tempdf=prioritized_mutations_control[i:i+1][0]
    control_df=control_df.merge(tempdf,left_index=True,right_index=True,how='outer')

    
control_df[0:5]
print(control_df.shape)
case_df=case_df.fillna(0)
control_df=control_df.fillna(0)

case_df['details']=case_df.index
control_df['details']=control_df.index
#
case_df['Chromosome']=case_df['details'].apply(lambda x : x.split('_')[0])
case_df['Position']=case_df['details'].apply(lambda x : x.split('_')[1])
case_df['Position1']=case_df['Position'].astype(int)+1
case_df['Reference allele']=case_df['details'].apply(lambda x : x.split('_')[2])
case_df['Alternate allele']=case_df['details'].apply(lambda x : x.split('_')[3])
#
control_df['Chromosome']=control_df['details'].apply(lambda x : x.split('_')[0])
control_df['Position']=control_df['details'].apply(lambda x : x.split('_')[1])
control_df['Position1']=control_df['Position'].astype(int)+1
control_df['Reference allele']=control_df['details'].apply(lambda x : x.split('_')[2])
control_df['Alternate allele']=control_df['details'].apply(lambda x : x.split('_')[3])

print(case_df[0:5])
print(control_df[0:5])



In [ ]:
####For Vignettes

In [ ]:
name='c1'
case_snps_in_peaks=pd.read_csv(
    'filtered_mutations/asd_affiltered_hg38_within_peaks_'+str(name)+'.bed',sep='\t',header=None)
case_snps_in_peaks.columns=['Chr','Pos0','Pos','rsid']
case_snps_in_peaks.shape
control_snps_in_peaks=pd.read_csv(
    'filtered_mutations/asd_ssc_control_affiltered_hg38_within_peaks_'+str(name)+'.bed',sep='\t',header=None)
control_snps_in_peaks.columns=['Chr','Pos0','Pos','rsid']
snps_inpeaks=case_snps_af_filtered_1.merge(case_snps_in_peaks,on=['Chr','Pos0','Pos','rsid'])
print(snps_inpeaks.shape)
control_snps_inpeaks=controls_snps_af_filtered_1.merge(control_snps_in_peaks,on=['Chr','Pos0','Pos','rsid'])
print(control_snps_inpeaks.shape)
snps_inpeaks.to_csv('filtered_mutations/asd_final_0base_'+str(name)+'.tsv',header=True,index=False,sep='\t')
control_snps_inpeaks.to_csv('filtered_mutations/asd_control_final_0base_'+str(name)+'.tsv',header=True,index=False,sep='\t')


#load the model! 
model_num=0

custom_objects={"recall":recall,
                    "sensitivity":recall,
                    "specificity":specificity,
                    "fpr":fpr,
                    "fnr":fnr,
                    "precision":precision,
                    "f1":f1,
                    "ambig_binary_crossentropy":ambig_binary_crossentropy,
                    "ambig_mean_absolute_error":ambig_mean_absolute_error,
                    "ambig_mean_squared_error":ambig_mean_squared_error,
                    "MultichannelMultinomialNLL":MultichannelMultinomialNLL}
get_custom_objects().update(custom_objects)

import json 
model=model_from_json(open("<google drive >"+name+"."+str(model_num)+".arch",'r').read())
model.load_weights("<google drive >"+name+"."+str(model_num)+".weights")




#reference allele sequence generator 
ref_gen=SNPGenerator(bed_path="filtered_mutations/asd_final_0base_"+name+".tsv",
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Ref",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)

#alternate allele sequence generator 
alt_gen=SNPGenerator(bed_path="filtered_mutations/asd_final_0base_"+name+".tsv",
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Alt",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)


#get the reference allele predictions 
count_preds={} 
profile_preds={} 
snp_to_seq={} 
for i in range(0,len(ref_gen)):
    print(str(i))
    cur_x=ref_gen[i] 
    batch_rsids=cur_x[0] 
    batch_preds=model.predict(cur_x[1])
    batch_preds_profile=batch_preds[0]
    batch_preds_count=batch_preds[1] 
    for batch_index in range(len(batch_rsids)): 
            cur_rsid=batch_rsids[batch_index]
            snp_to_seq[cur_rsid]={} 
            snp_to_seq[cur_rsid]['ref']=cur_x[1][batch_index,:,:]
            cur_pred_profile=batch_preds_profile[batch_index,:,:]
            cur_pred_count=batch_preds_count[batch_index,:][0]
            count_preds[cur_rsid]={}
            count_preds[cur_rsid]['ref']=cur_pred_count
            profile_preds[cur_rsid]={}
            profile_preds[cur_rsid]['ref']=cur_pred_profile 
            


#get the alternate allele predictions 
for i in range(0,len(alt_gen)):
    print(str(i))
    cur_x=alt_gen[i] 
    batch_rsids=cur_x[0] 
    batch_preds=model.predict(cur_x[1])
    batch_preds_profile=batch_preds[0]
    batch_preds_count=batch_preds[1] 
    for batch_index in range(len(batch_rsids)): 
            cur_rsid=batch_rsids[batch_index]
            snp_to_seq[cur_rsid]['alt']=cur_x[1][batch_index,:,:]
            cur_pred_profile=batch_preds_profile[batch_index,:,:]
            cur_pred_count=batch_preds_count[batch_index,:][0]
            count_preds[cur_rsid]['alt']=cur_pred_count
            profile_preds[cur_rsid]['alt']=cur_pred_profile 
    
    
count_preds_cases=count_preds
profile_preds_cases=profile_preds
snp_to_seq_cases=snp_to_seq

log_odd_changes_sum_10=[]
log_odd_changes_sum_20=[]
log_odd_changes_sum_50=[]
log_odd_changes_sum_100=[]
log_odd_changes_sum_150=[]
log_odd_changes_sum_200=[]
log_odd_changes_sum_250=[]
log_odd_changes_sum_300=[]
log_odd_changes_sum_1000=[]
for i in profile_preds_cases.keys():
    pred_prob_ref=softmax(profile_preds_cases[i]['ref'],axis=0)
    pred_prob_alt=softmax(profile_preds_cases[i]['alt'],axis=0)
    log_odd_changes_sum_1000.append(np.sum(np.log(pred_prob_ref)-np.log(pred_prob_alt)))
    log_odd_changes_sum_10.append(np.sum(np.log(pred_prob_ref[495:505])-np.log(pred_prob_alt[495:505])))
    log_odd_changes_sum_20.append(np.sum(np.log(pred_prob_ref[490:510])-np.log(pred_prob_alt[490:510])))
    log_odd_changes_sum_50.append(np.sum(np.log(pred_prob_ref[475:525])-np.log(pred_prob_alt[475:525])))
    log_odd_changes_sum_100.append(np.sum(np.log(pred_prob_ref[450:550])-np.log(pred_prob_alt[450:550])))
    log_odd_changes_sum_150.append(np.sum(np.log(pred_prob_ref[425:575])-np.log(pred_prob_alt[425:575])))
    log_odd_changes_sum_200.append(np.sum(np.log(pred_prob_ref[400:600])-np.log(pred_prob_alt[400:600])))
    log_odd_changes_sum_250.append(np.sum(np.log(pred_prob_ref[375:625])-np.log(pred_prob_alt[375:625])))
    log_odd_changes_sum_300.append(np.sum(np.log(pred_prob_ref[350:650])-np.log(pred_prob_alt[350:650])))

    
local_count_changes_10=[]
local_count_changes_20=[]
local_count_changes_50=[]
local_count_changes_100=[]
local_count_changes_150=[]
local_count_changes_200=[]
local_count_changes_250=[]
local_count_changes_300=[]
local_count_changes_1000=[]

for i in profile_preds_cases.keys():
    pred_prob_ref=softmax(profile_preds_cases[i]['ref'],axis=0)
    pred_prob_alt=softmax(profile_preds_cases[i]['alt'],axis=0)
    count_pred_ref=count_preds_cases[i]['ref']
    count_pred_alt=count_preds_cases[i]['alt']
    local_count_changes_1000.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref-np.exp(count_pred_alt)*pred_prob_alt))
    local_count_changes_10.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[495:505]-np.exp(count_pred_alt)*pred_prob_alt[495:505]))
    local_count_changes_20.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[490:510]-np.exp(count_pred_alt)*pred_prob_alt[490:510]))
    local_count_changes_50.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[475:525]-np.exp(count_pred_alt)*pred_prob_alt[475:525]))
    local_count_changes_100.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[450:550]-np.exp(count_pred_alt)*pred_prob_alt[450:550]))
    local_count_changes_150.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[425:575]-np.exp(count_pred_alt)*pred_prob_alt[425:575]))
    local_count_changes_200.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[400:600]-np.exp(count_pred_alt)*pred_prob_alt[400:600]))
    local_count_changes_250.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[375:625]-np.exp(count_pred_alt)*pred_prob_alt[375:625]))
    local_count_changes_300.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[350:650]-np.exp(count_pred_alt)*pred_prob_alt[350:650]))
   


profile_preds_df=pd.DataFrame.from_dict(profile_preds_cases,orient='index')
profile_preds_df['log_odd_changes_sum_10']=log_odd_changes_sum_10
profile_preds_df['log_odd_changes_sum_20']=log_odd_changes_sum_20
profile_preds_df['log_odd_changes_sum_50']=log_odd_changes_sum_50
profile_preds_df['log_odd_changes_sum_100']=log_odd_changes_sum_100
profile_preds_df['log_odd_changes_sum_150']=log_odd_changes_sum_150
profile_preds_df['log_odd_changes_sum_200']=log_odd_changes_sum_200
profile_preds_df['log_odd_changes_sum_250']=log_odd_changes_sum_250
profile_preds_df['log_odd_changes_sum_300']=log_odd_changes_sum_300
profile_preds_df['log_odd_changes_sum_1000']=log_odd_changes_sum_1000

profile_preds_df['local_count_changes_10']=local_count_changes_10
profile_preds_df['local_count_changes_20']=local_count_changes_20
profile_preds_df['local_count_changes_50']=local_count_changes_50
profile_preds_df['local_count_changes_100']=local_count_changes_100
profile_preds_df['local_count_changes_150']=local_count_changes_150
profile_preds_df['local_count_changes_200']=local_count_changes_200
profile_preds_df['local_count_changes_250']=local_count_changes_250
profile_preds_df['local_count_changes_300']=local_count_changes_300
profile_preds_df['local_count_changes_1000']=local_count_changes_1000



#reference allele sequence generator 
ref_gen=SNPGenerator(bed_path='filtered_mutations/asd_control_final_0base_'+str(name)+'.tsv',
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Ref",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherrfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)

#alternate allele sequence generator 
alt_gen=SNPGenerator(bed_path='filtered_mutations/asd_control_final_0base_'+str(name)+'.tsv',
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Alt",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)




#get the reference allele predictions 
count_preds={} 
profile_preds={} 
snp_to_seq={} 
for i in range(0,len(ref_gen)):
    print(str(i))
    cur_x=ref_gen[i] 
    batch_rsids=cur_x[0] 
    batch_preds=model.predict(cur_x[1])
    batch_preds_profile=batch_preds[0]
    batch_preds_count=batch_preds[1] 
    for batch_index in range(len(batch_rsids)): 
            cur_rsid=batch_rsids[batch_index]
            snp_to_seq[cur_rsid]={} 
            snp_to_seq[cur_rsid]['ref']=cur_x[1][batch_index,:,:]
            cur_pred_profile=batch_preds_profile[batch_index,:,:]
            cur_pred_count=batch_preds_count[batch_index,:][0]
            count_preds[cur_rsid]={}
            count_preds[cur_rsid]['ref']=cur_pred_count
            profile_preds[cur_rsid]={}
            profile_preds[cur_rsid]['ref']=cur_pred_profile 
            

            
#get the alternate allele predictions 
for i in range(0,len(alt_gen)):
    print(str(i))
    cur_x=alt_gen[i] 
    batch_rsids=cur_x[0] 
    batch_preds=model.predict(cur_x[1])
    batch_preds_profile=batch_preds[0]
    batch_preds_count=batch_preds[1] 
    for batch_index in range(len(batch_rsids)): 
            cur_rsid=batch_rsids[batch_index]
            snp_to_seq[cur_rsid]['alt']=cur_x[1][batch_index,:,:]
            cur_pred_profile=batch_preds_profile[batch_index,:,:]
            cur_pred_count=batch_preds_count[batch_index,:][0]
            count_preds[cur_rsid]['alt']=cur_pred_count
            profile_preds[cur_rsid]['alt']=cur_pred_profile 
    

    
count_preds_controls=count_preds
profile_preds_controls=profile_preds
snp_to_seq_controls=snp_to_seq



log_odd_changes_sum_10=[]
log_odd_changes_sum_20=[]
log_odd_changes_sum_50=[]
log_odd_changes_sum_100=[]
log_odd_changes_sum_150=[]
log_odd_changes_sum_200=[]
log_odd_changes_sum_250=[]
log_odd_changes_sum_300=[]
log_odd_changes_sum_1000=[]
for i in profile_preds_controls.keys():
    pred_prob_ref=softmax(profile_preds_controls[i]['ref'],axis=0)
    pred_prob_alt=softmax(profile_preds_controls[i]['alt'],axis=0)
    log_odd_changes_sum_1000.append(np.sum(np.log(pred_prob_ref)-np.log(pred_prob_alt)))
    log_odd_changes_sum_10.append(np.sum(np.log(pred_prob_ref[495:505])-np.log(pred_prob_alt[495:505])))
    log_odd_changes_sum_20.append(np.sum(np.log(pred_prob_ref[490:510])-np.log(pred_prob_alt[490:510])))
    log_odd_changes_sum_50.append(np.sum(np.log(pred_prob_ref[475:525])-np.log(pred_prob_alt[475:525])))
    log_odd_changes_sum_100.append(np.sum(np.log(pred_prob_ref[450:550])-np.log(pred_prob_alt[450:550])))
    log_odd_changes_sum_150.append(np.sum(np.log(pred_prob_ref[425:575])-np.log(pred_prob_alt[425:575])))
    log_odd_changes_sum_200.append(np.sum(np.log(pred_prob_ref[400:600])-np.log(pred_prob_alt[400:600])))
    log_odd_changes_sum_250.append(np.sum(np.log(pred_prob_ref[375:625])-np.log(pred_prob_alt[375:625])))
    log_odd_changes_sum_300.append(np.sum(np.log(pred_prob_ref[350:650])-np.log(pred_prob_alt[350:650])))

    
    
local_count_changes_10=[]
local_count_changes_20=[]
local_count_changes_50=[]
local_count_changes_100=[]
local_count_changes_150=[]
local_count_changes_200=[]
local_count_changes_250=[]
local_count_changes_300=[]
local_count_changes_1000=[]
for i in profile_preds_controls.keys():
    pred_prob_ref=softmax(profile_preds_controls[i]['ref'],axis=0)
    pred_prob_alt=softmax(profile_preds_controls[i]['alt'],axis=0)
    count_pred_ref=count_preds_controls[i]['ref']
    count_pred_alt=count_preds_controls[i]['alt']
    local_count_changes_1000.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref-np.exp(count_pred_alt)*pred_prob_alt))
    local_count_changes_10.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[495:505]-np.exp(count_pred_alt)*pred_prob_alt[495:505]))
    local_count_changes_20.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[490:510]-np.exp(count_pred_alt)*pred_prob_alt[490:510]))
    local_count_changes_50.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[475:525]-np.exp(count_pred_alt)*pred_prob_alt[475:525]))
    local_count_changes_100.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[450:550]-np.exp(count_pred_alt)*pred_prob_alt[450:550]))
    local_count_changes_150.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[425:575]-np.exp(count_pred_alt)*pred_prob_alt[425:575]))
    local_count_changes_200.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[400:600]-np.exp(count_pred_alt)*pred_prob_alt[400:600]))
    local_count_changes_250.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[375:625]-np.exp(count_pred_alt)*pred_prob_alt[375:625]))
    local_count_changes_300.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[350:650]-np.exp(count_pred_alt)*pred_prob_alt[350:650]))
    


profile_preds_df_controls=pd.DataFrame.from_dict(profile_preds_controls,orient='index')
profile_preds_df_controls['log_odd_changes_sum_10']=log_odd_changes_sum_10
profile_preds_df_controls['log_odd_changes_sum_20']=log_odd_changes_sum_20
profile_preds_df_controls['log_odd_changes_sum_50']=log_odd_changes_sum_50
profile_preds_df_controls['log_odd_changes_sum_100']=log_odd_changes_sum_100
profile_preds_df_controls['log_odd_changes_sum_150']=log_odd_changes_sum_150
profile_preds_df_controls['log_odd_changes_sum_200']=log_odd_changes_sum_200
profile_preds_df_controls['log_odd_changes_sum_250']=log_odd_changes_sum_250
profile_preds_df_controls['log_odd_changes_sum_300']=log_odd_changes_sum_300
profile_preds_df_controls['log_odd_changes_sum_1000']=log_odd_changes_sum_1000

profile_preds_df_controls['local_count_changes_10']=local_count_changes_10
profile_preds_df_controls['local_count_changes_20']=local_count_changes_20
profile_preds_df_controls['local_count_changes_50']=local_count_changes_50
profile_preds_df_controls['local_count_changes_100']=local_count_changes_100
profile_preds_df_controls['local_count_changes_150']=local_count_changes_150
profile_preds_df_controls['local_count_changes_200']=local_count_changes_200
profile_preds_df_controls['local_count_changes_250']=local_count_changes_250
profile_preds_df_controls['local_count_changes_300']=local_count_changes_300
profile_preds_df_controls['local_count_changes_1000']=local_count_changes_1000


In [ ]:
#get shap values for the SNPs with the highest delta 

model_wrapper=(model.input, model.layers[-1].output)
count_explainer=shap.DeepExplainer(model_wrapper,data=create_background,combine_mult_and_diffref=combine_mult_and_diffref_1d)
prof_explainer = create_explainer(model,ischip=False)

In [ ]:
count_preds=count_preds_cases
profile_preds=profile_preds_cases
snp_to_seq=snp_to_seq_cases


In [ ]:
#mutation position of interest
a='chr1_60897164_G_A'
a1=a.split('_')[0]
a2=int(a.split('_')[1])
a2
#get the signal in probability space 
from sklearn.preprocessing import normalize
signal=np.nan_to_num(bw.values(a1,a2-500,a2+500))
signal=signal/np.sum(signal)
print(signal.shape)


In [ ]:

#let's explain chr6_161153273_C_T
rsid=a
print(a)
#signal=math.log(sum(pbw.values(a1,a2-500,a2+500)))
cur_seq_ref=np.expand_dims(snp_to_seq[rsid]['ref'],axis=0)
cur_seq_alt=np.expand_dims(snp_to_seq[rsid]['alt'],axis=0)
count_pred_ref=count_preds[rsid]['ref']
count_pred_alt=count_preds[rsid]['alt']
pred_ref=profile_preds[rsid]['ref']
pred_alt=profile_preds[rsid]['alt']
pred_prob_ref=softmax(profile_preds[rsid]['ref'],axis=0)
pred_prob_alt=softmax(profile_preds[rsid]['alt'],axis=0)
profile_explanations_ref=prof_explainer(cur_seq_ref,None)
count_explanations_ref=np.squeeze(count_explainer.shap_values(cur_seq_ref)[0])
profile_explanations_alt=prof_explainer(cur_seq_alt,None)
count_explanations_alt=np.squeeze(count_explainer.shap_values(cur_seq_alt)[0])
print(np.sum(profile_explanations_ref[0,1047:1067,:]-profile_explanations_alt[0,1047:1067,:]))
print(np.sum(profile_explanations_ref[0,1047:1067,:]))
print(np.sum(profile_explanations_alt[0,1047:1067,:]))


In [ ]:
## for plotting 
%load_ext autoreload
%reload_ext autoreload

%autoreload 2
%matplotlib inline
import matplotlib 
from matplotlib import pyplot as plt
plt.rcParams["figure.figsize"]=10,5
plt.rcParams['axes.xmargin'] = 0

font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 10}

matplotlib.rc('font', **font)
matplotlib.rc('font', **font)
matplotlib.rcParams['pdf.fonttype']=42


In [ ]:

def make_plot(rsid,
              count_pred_ref,
              count_pred_alt,
              pred_prob_ref,
              pred_prob_alt,
              profile_explanations_ref, 
              seq_ref,
              profile_explanations_alt,
              seq_alt,
              count_explanations_ref, 
              count_explanations_alt,
              xmin,
              xmax,
              ymin_prof_shap=-0.1,
              ymin_count_shap=-0.1,
              ymin_perf=-0.1,
              ymax_prof_shap=0.1,
              ymax_count_shap=0.1,
              ymax_perf=0.1):
    plt.rcParams["figure.figsize"]=20,10
    matplotlib.rcParams['pdf.fonttype']=42
    fig, axes = plt.subplots(7, 1)
    axes[0].plot(np.exp(count_pred_ref)*pred_prob_ref,label='Ref pred counts per base',color='b')
    axes[0].plot(np.exp(count_pred_alt)*pred_prob_alt,label='Alt pred counts per base',color='r')
    axes[0].set_title(str(rsid)+"Counts Pred Ref:"+str(count_pred_ref)+":"+"Counts Pred Alt:"+str(count_pred_alt))
    axes[0].legend() 
    axes[0].set_xlim(xmin,xmax)
    #axes[0].set_ylim(ymin_perf,ymax_perf)
    axes[0].set_xticks(list(range(xmin, xmax, 50,)))    
    #
    #axes[1].plot(signal,label='Label Prob',color='g')
    axes[1].plot(np.log(pred_prob_ref)-np.log(pred_prob_alt),label='Ref prob - alt prob',color='k')
    axes[1].legend() 
    #axes[1].set_xlim(xmin,xmax)
    #axes[1].set_ylim(ymin_perf,ymax_perf)
    axes[1].set_xticks(list(range(xmin, xmax, 50,)))    
    #
    axes[2].plot(pred_prob_ref,label='Ref prob',color='r')
    axes[2].plot(pred_prob_alt,label='Alt prob',color='b',alpha=0.5)
    axes[2].legend() 
    #axes[2].set_xlim(xmin,xmax)
    #axes[2].set_ylim(ymin_perf,ymax_perf)
    axes[2].set_xticks(list(range(xmin, xmax, 50,)))    
    #
    axes[3]=plot_seq_importance(profile_explanations_ref,seq_ref,xlim=(xmin,xmax),axes=axes[3])
    axes[3].set_title("Profile Loss SHAP, Ref")        
    axes[3].set_ylim(ymin_prof_shap,ymax_prof_shap)
    axes[3].set_xticks(list(range(xmin, xmax, 50,)))  
    #
    axes[4]=plot_seq_importance(profile_explanations_alt,seq_alt,xlim=(xmin,xmax),axes=axes[4])
    axes[4].set_title("Profile Loss SHAP, Alt")        
    axes[4].set_ylim(ymin_prof_shap,ymax_prof_shap)
    axes[4].set_xticks(list(range(xmin, xmax, 50,)))    
    #   
    axes[5]=plot_seq_importance(count_explanations_ref,seq_ref,xlim=(xmin,xmax),axes=axes[5])
    axes[5].set_title("Count Loss SHAP, Ref")        
    axes[5].set_ylim(ymin_count_shap,ymax_count_shap)
    axes[5].set_xticks(list(range(xmin, xmax, 50,)))    
    #
    axes[6]=plot_seq_importance(count_explanations_alt,seq_alt,xlim=(xmin,xmax),axes=axes[6])
    axes[6].set_title("Count Loss SHAP, Alt")        
    axes[6].set_ylim(ymin_count_shap,ymax_count_shap)
    axes[6].set_xticks(list(range(xmin, xmax, 50,))) 
    #
    plt.subplots_adjust(hspace=0.6)
    plt.show()

In [ ]:
make_plot(rsid,
          count_pred_ref,
          count_pred_alt,
          pred_prob_ref,
          pred_prob_alt,
          profile_explanations_ref[:,557:557+1000,:],
          cur_seq_ref[:,557:557+1000,:],
          profile_explanations_alt[:,557:557+1000,:],
          cur_seq_alt[:,557:557+1000,:],
          count_explanations_ref[557:557+1000,:],
          count_explanations_alt[557:557+1000,:],
          xmin=0,
          xmax=1000,
          ymin_prof_shap=-0.02,
          ymin_count_shap=-0.02,
          ymin_perf=-0.02,
          ymax_prof_shap=0.06,
          ymax_count_shap=0.04,
          ymax_perf=0.02,
         )





In [ ]:

make_plot(rsid,
          count_pred_ref,
          count_pred_alt,
          pred_prob_ref,
          pred_prob_alt,
          profile_explanations_ref[:,557:557+1000,:],
          cur_seq_ref[:,557:557+1000,:],
          profile_explanations_alt[:,557:557+1000,:],
          cur_seq_alt[:,557:557+1000,:],
          count_explanations_ref[557:557+1000,:],
          count_explanations_alt[557:557+1000,:],
          xmin=450,
          xmax=550,
          ymin_prof_shap=-0.02,
          ymin_count_shap=-0.02,
          ymin_perf=-0.02,
          ymax_prof_shap=0.05,
          ymax_count_shap=0.05,
          ymax_perf=0.1)



In [ ]:
#### motif enrichment analysis

In [ ]:
case_motif=pd.read_csv('motif_files/case_motifs.bed',sep='\t',header=None)
print(case_motif[0:5])
control_motif=pd.read_csv('motif_files/control_motifs.bed',sep='\t',header=None)
print(control_motif[0:5])


In [ ]:
case_motif1=case_motif[case_motif[9].str.contains('_MA')].merge(case_df,on=['Chromosome','Position','Position1'])
control_motif1=control_motif[control_motif[9].str.contains('_MA')].merge(control_df,on=['Chromosome','Position','Position1'])

In [ ]:
###insert the required cluster name here
cluster_case_motif1=case_motif1[case_motif1['details'].isin(case_motif1[case_motif1.iloc[:,12:25].idxmax(axis=1)=='<clustername>_cases_100_odds']['details'])]
print(cluster_case_motif1.shape)
cluster_control_motif1=control_motif1[control_motif1['details'].isin(control_motif1[control_motif1.iloc[:,12:25].idxmax(axis=1)=='<clustername>_cases_100_odds']['details'])]
print(cluster_control_motif1.shape)



In [ ]:
#load the cluster specific model
name='clustername'
case_snps_in_peaks=pd.read_csv(
    'filtered_mutations/asd_affiltered_hg38_within_peaks_'+str(name)+'.bed',sep='\t',header=None)
case_snps_in_peaks.columns=['Chr','Pos0','Pos','rsid']
case_snps_in_peaks.shape
control_snps_in_peaks=pd.read_csv(
    'filtered_mutations/asd_ssc_control_affiltered_hg38_within_peaks_'+str(name)+'.bed',sep='\t',header=None)
control_snps_in_peaks.columns=['Chr','Pos0','Pos','rsid']
snps_inpeaks=case_snps_af_filtered_1.merge(case_snps_in_peaks,on=['Chr','Pos0','Pos','rsid'])
print(snps_inpeaks.shape)
control_snps_inpeaks=controls_snps_af_filtered_1.merge(control_snps_in_peaks,on=['Chr','Pos0','Pos','rsid'])
print(control_snps_inpeaks.shape)
snps_inpeaks.to_csv('filtered_mutations/asd_final_0base_'+str(name)+'.tsv',header=True,index=False,sep='\t')
control_snps_inpeaks.to_csv('filtered_mutations/asd_control_final_0base_'+str(name)+'.tsv',header=True,index=False,sep='\t')


#load the model! 
model_num=0

custom_objects={"recall":recall,
                    "sensitivity":recall,
                    "specificity":specificity,
                    "fpr":fpr,
                    "fnr":fnr,
                    "precision":precision,
                    "f1":f1,
                    "ambig_binary_crossentropy":ambig_binary_crossentropy,
                    "ambig_mean_absolute_error":ambig_mean_absolute_error,
                    "ambig_mean_squared_error":ambig_mean_squared_error,
                    "MultichannelMultinomialNLL":MultichannelMultinomialNLL}
get_custom_objects().update(custom_objects)

import json 
model=model_from_json(open("<google dive>"+name+"."+str(model_num)+".arch",'r').read())
model.load_weights("<google drive>"+name+"."+str(model_num)+".weights")




#reference allele sequence generator 
ref_gen=SNPGenerator(bed_path="filtered_mutations/asd_final_0base_"+name+".tsv",
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Ref",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)

#alternate allele sequence generator 
alt_gen=SNPGenerator(bed_path="filtered_mutations/asd_final_0base_"+name+".tsv",
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Alt",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)


#get the reference allele predictions 
count_preds={} 
profile_preds={} 
snp_to_seq={} 
for i in range(0,len(ref_gen)):
    print(str(i))
    cur_x=ref_gen[i] 
    batch_rsids=cur_x[0] 
    batch_preds=model.predict(cur_x[1])
    batch_preds_profile=batch_preds[0]
    batch_preds_count=batch_preds[1] 
    for batch_index in range(len(batch_rsids)): 
            cur_rsid=batch_rsids[batch_index]
            snp_to_seq[cur_rsid]={} 
            snp_to_seq[cur_rsid]['ref']=cur_x[1][batch_index,:,:]
            cur_pred_profile=batch_preds_profile[batch_index,:,:]
            cur_pred_count=batch_preds_count[batch_index,:][0]
            count_preds[cur_rsid]={}
            count_preds[cur_rsid]['ref']=cur_pred_count
            profile_preds[cur_rsid]={}
            profile_preds[cur_rsid]['ref']=cur_pred_profile 
            


#get the alternate allele predictions 
for i in range(0,len(alt_gen)):
    print(str(i))
    cur_x=alt_gen[i] 
    batch_rsids=cur_x[0] 
    batch_preds=model.predict(cur_x[1])
    batch_preds_profile=batch_preds[0]
    batch_preds_count=batch_preds[1] 
    for batch_index in range(len(batch_rsids)): 
            cur_rsid=batch_rsids[batch_index]
            snp_to_seq[cur_rsid]['alt']=cur_x[1][batch_index,:,:]
            cur_pred_profile=batch_preds_profile[batch_index,:,:]
            cur_pred_count=batch_preds_count[batch_index,:][0]
            count_preds[cur_rsid]['alt']=cur_pred_count
            profile_preds[cur_rsid]['alt']=cur_pred_profile 
    
    
count_preds_cases=count_preds
profile_preds_cases=profile_preds
snp_to_seq_cases=snp_to_seq

log_odd_changes_sum_10=[]
log_odd_changes_sum_20=[]
log_odd_changes_sum_50=[]
log_odd_changes_sum_100=[]
log_odd_changes_sum_150=[]
log_odd_changes_sum_200=[]
log_odd_changes_sum_250=[]
log_odd_changes_sum_300=[]
log_odd_changes_sum_1000=[]
for i in profile_preds_cases.keys():
    pred_prob_ref=softmax(profile_preds_cases[i]['ref'],axis=0)
    pred_prob_alt=softmax(profile_preds_cases[i]['alt'],axis=0)
    log_odd_changes_sum_1000.append(np.sum(np.log(pred_prob_ref)-np.log(pred_prob_alt)))
    log_odd_changes_sum_10.append(np.sum(np.log(pred_prob_ref[495:505])-np.log(pred_prob_alt[495:505])))
    log_odd_changes_sum_20.append(np.sum(np.log(pred_prob_ref[490:510])-np.log(pred_prob_alt[490:510])))
    log_odd_changes_sum_50.append(np.sum(np.log(pred_prob_ref[475:525])-np.log(pred_prob_alt[475:525])))
    log_odd_changes_sum_100.append(np.sum(np.log(pred_prob_ref[450:550])-np.log(pred_prob_alt[450:550])))
    log_odd_changes_sum_150.append(np.sum(np.log(pred_prob_ref[425:575])-np.log(pred_prob_alt[425:575])))
    log_odd_changes_sum_200.append(np.sum(np.log(pred_prob_ref[400:600])-np.log(pred_prob_alt[400:600])))
    log_odd_changes_sum_250.append(np.sum(np.log(pred_prob_ref[375:625])-np.log(pred_prob_alt[375:625])))
    log_odd_changes_sum_300.append(np.sum(np.log(pred_prob_ref[350:650])-np.log(pred_prob_alt[350:650])))

    
local_count_changes_10=[]
local_count_changes_20=[]
local_count_changes_50=[]
local_count_changes_100=[]
local_count_changes_150=[]
local_count_changes_200=[]
local_count_changes_250=[]
local_count_changes_300=[]
local_count_changes_1000=[]

for i in profile_preds_cases.keys():
    pred_prob_ref=softmax(profile_preds_cases[i]['ref'],axis=0)
    pred_prob_alt=softmax(profile_preds_cases[i]['alt'],axis=0)
    count_pred_ref=count_preds_cases[i]['ref']
    count_pred_alt=count_preds_cases[i]['alt']
    local_count_changes_1000.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref-np.exp(count_pred_alt)*pred_prob_alt))
    local_count_changes_10.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[495:505]-np.exp(count_pred_alt)*pred_prob_alt[495:505]))
    local_count_changes_20.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[490:510]-np.exp(count_pred_alt)*pred_prob_alt[490:510]))
    local_count_changes_50.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[475:525]-np.exp(count_pred_alt)*pred_prob_alt[475:525]))
    local_count_changes_100.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[450:550]-np.exp(count_pred_alt)*pred_prob_alt[450:550]))
    local_count_changes_150.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[425:575]-np.exp(count_pred_alt)*pred_prob_alt[425:575]))
    local_count_changes_200.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[400:600]-np.exp(count_pred_alt)*pred_prob_alt[400:600]))
    local_count_changes_250.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[375:625]-np.exp(count_pred_alt)*pred_prob_alt[375:625]))
    local_count_changes_300.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[350:650]-np.exp(count_pred_alt)*pred_prob_alt[350:650]))
   


profile_preds_df=pd.DataFrame.from_dict(profile_preds_cases,orient='index')
profile_preds_df['log_odd_changes_sum_10']=log_odd_changes_sum_10
profile_preds_df['log_odd_changes_sum_20']=log_odd_changes_sum_20
profile_preds_df['log_odd_changes_sum_50']=log_odd_changes_sum_50
profile_preds_df['log_odd_changes_sum_100']=log_odd_changes_sum_100
profile_preds_df['log_odd_changes_sum_150']=log_odd_changes_sum_150
profile_preds_df['log_odd_changes_sum_200']=log_odd_changes_sum_200
profile_preds_df['log_odd_changes_sum_250']=log_odd_changes_sum_250
profile_preds_df['log_odd_changes_sum_300']=log_odd_changes_sum_300
profile_preds_df['log_odd_changes_sum_1000']=log_odd_changes_sum_1000

profile_preds_df['local_count_changes_10']=local_count_changes_10
profile_preds_df['local_count_changes_20']=local_count_changes_20
profile_preds_df['local_count_changes_50']=local_count_changes_50
profile_preds_df['local_count_changes_100']=local_count_changes_100
profile_preds_df['local_count_changes_150']=local_count_changes_150
profile_preds_df['local_count_changes_200']=local_count_changes_200
profile_preds_df['local_count_changes_250']=local_count_changes_250
profile_preds_df['local_count_changes_300']=local_count_changes_300
profile_preds_df['local_count_changes_1000']=local_count_changes_1000



#reference allele sequence generator 
ref_gen=SNPGenerator(bed_path='filtered_mutations/asd_control_final_0base_'+str(name)+'.tsv',
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Ref",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)

#alternate allele sequence generator 
alt_gen=SNPGenerator(bed_path='filtered_mutations/asd_control_final_0base_'+str(name)+'.tsv',
                 chrom_col="Chr",
                 pos_col="Pos0",
                 allele_col="Alt",
                 rsid_col='rsid',
                 flank_size=1057,
                 ref_fasta="Otherrfiles/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta",
                 expand_dims=False)




#get the reference allele predictions 
count_preds={} 
profile_preds={} 
snp_to_seq={} 
for i in range(0,len(ref_gen)):
    print(str(i))
    cur_x=ref_gen[i] 
    batch_rsids=cur_x[0] 
    batch_preds=model.predict(cur_x[1])
    batch_preds_profile=batch_preds[0]
    batch_preds_count=batch_preds[1] 
    for batch_index in range(len(batch_rsids)): 
            cur_rsid=batch_rsids[batch_index]
            snp_to_seq[cur_rsid]={} 
            snp_to_seq[cur_rsid]['ref']=cur_x[1][batch_index,:,:]
            cur_pred_profile=batch_preds_profile[batch_index,:,:]
            cur_pred_count=batch_preds_count[batch_index,:][0]
            count_preds[cur_rsid]={}
            count_preds[cur_rsid]['ref']=cur_pred_count
            profile_preds[cur_rsid]={}
            profile_preds[cur_rsid]['ref']=cur_pred_profile 
            

            
#get the alternate allele predictions 
for i in range(0,len(alt_gen)):
    print(str(i))
    cur_x=alt_gen[i] 
    batch_rsids=cur_x[0] 
    batch_preds=model.predict(cur_x[1])
    batch_preds_profile=batch_preds[0]
    batch_preds_count=batch_preds[1] 
    for batch_index in range(len(batch_rsids)): 
            cur_rsid=batch_rsids[batch_index]
            snp_to_seq[cur_rsid]['alt']=cur_x[1][batch_index,:,:]
            cur_pred_profile=batch_preds_profile[batch_index,:,:]
            cur_pred_count=batch_preds_count[batch_index,:][0]
            count_preds[cur_rsid]['alt']=cur_pred_count
            profile_preds[cur_rsid]['alt']=cur_pred_profile 
    

    
count_preds_controls=count_preds
profile_preds_controls=profile_preds
snp_to_seq_controls=snp_to_seq



log_odd_changes_sum_10=[]
log_odd_changes_sum_20=[]
log_odd_changes_sum_50=[]
log_odd_changes_sum_100=[]
log_odd_changes_sum_150=[]
log_odd_changes_sum_200=[]
log_odd_changes_sum_250=[]
log_odd_changes_sum_300=[]
log_odd_changes_sum_1000=[]
for i in profile_preds_controls.keys():
    pred_prob_ref=softmax(profile_preds_controls[i]['ref'],axis=0)
    pred_prob_alt=softmax(profile_preds_controls[i]['alt'],axis=0)
    log_odd_changes_sum_1000.append(np.sum(np.log(pred_prob_ref)-np.log(pred_prob_alt)))
    log_odd_changes_sum_10.append(np.sum(np.log(pred_prob_ref[495:505])-np.log(pred_prob_alt[495:505])))
    log_odd_changes_sum_20.append(np.sum(np.log(pred_prob_ref[490:510])-np.log(pred_prob_alt[490:510])))
    log_odd_changes_sum_50.append(np.sum(np.log(pred_prob_ref[475:525])-np.log(pred_prob_alt[475:525])))
    log_odd_changes_sum_100.append(np.sum(np.log(pred_prob_ref[450:550])-np.log(pred_prob_alt[450:550])))
    log_odd_changes_sum_150.append(np.sum(np.log(pred_prob_ref[425:575])-np.log(pred_prob_alt[425:575])))
    log_odd_changes_sum_200.append(np.sum(np.log(pred_prob_ref[400:600])-np.log(pred_prob_alt[400:600])))
    log_odd_changes_sum_250.append(np.sum(np.log(pred_prob_ref[375:625])-np.log(pred_prob_alt[375:625])))
    log_odd_changes_sum_300.append(np.sum(np.log(pred_prob_ref[350:650])-np.log(pred_prob_alt[350:650])))

    
    
local_count_changes_10=[]
local_count_changes_20=[]
local_count_changes_50=[]
local_count_changes_100=[]
local_count_changes_150=[]
local_count_changes_200=[]
local_count_changes_250=[]
local_count_changes_300=[]
local_count_changes_1000=[]
for i in profile_preds_controls.keys():
    pred_prob_ref=softmax(profile_preds_controls[i]['ref'],axis=0)
    pred_prob_alt=softmax(profile_preds_controls[i]['alt'],axis=0)
    count_pred_ref=count_preds_controls[i]['ref']
    count_pred_alt=count_preds_controls[i]['alt']
    local_count_changes_1000.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref-np.exp(count_pred_alt)*pred_prob_alt))
    local_count_changes_10.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[495:505]-np.exp(count_pred_alt)*pred_prob_alt[495:505]))
    local_count_changes_20.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[490:510]-np.exp(count_pred_alt)*pred_prob_alt[490:510]))
    local_count_changes_50.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[475:525]-np.exp(count_pred_alt)*pred_prob_alt[475:525]))
    local_count_changes_100.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[450:550]-np.exp(count_pred_alt)*pred_prob_alt[450:550]))
    local_count_changes_150.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[425:575]-np.exp(count_pred_alt)*pred_prob_alt[425:575]))
    local_count_changes_200.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[400:600]-np.exp(count_pred_alt)*pred_prob_alt[400:600]))
    local_count_changes_250.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[375:625]-np.exp(count_pred_alt)*pred_prob_alt[375:625]))
    local_count_changes_300.append(np.sum(np.exp(count_pred_ref)*pred_prob_ref[350:650]-np.exp(count_pred_alt)*pred_prob_alt[350:650]))
    


profile_preds_df_controls=pd.DataFrame.from_dict(profile_preds_controls,orient='index')
profile_preds_df_controls['log_odd_changes_sum_10']=log_odd_changes_sum_10
profile_preds_df_controls['log_odd_changes_sum_20']=log_odd_changes_sum_20
profile_preds_df_controls['log_odd_changes_sum_50']=log_odd_changes_sum_50
profile_preds_df_controls['log_odd_changes_sum_100']=log_odd_changes_sum_100
profile_preds_df_controls['log_odd_changes_sum_150']=log_odd_changes_sum_150
profile_preds_df_controls['log_odd_changes_sum_200']=log_odd_changes_sum_200
profile_preds_df_controls['log_odd_changes_sum_250']=log_odd_changes_sum_250
profile_preds_df_controls['log_odd_changes_sum_300']=log_odd_changes_sum_300
profile_preds_df_controls['log_odd_changes_sum_1000']=log_odd_changes_sum_1000

profile_preds_df_controls['local_count_changes_10']=local_count_changes_10
profile_preds_df_controls['local_count_changes_20']=local_count_changes_20
profile_preds_df_controls['local_count_changes_50']=local_count_changes_50
profile_preds_df_controls['local_count_changes_100']=local_count_changes_100
profile_preds_df_controls['local_count_changes_150']=local_count_changes_150
profile_preds_df_controls['local_count_changes_200']=local_count_changes_200
profile_preds_df_controls['local_count_changes_250']=local_count_changes_250
profile_preds_df_controls['local_count_changes_300']=local_count_changes_300
profile_preds_df_controls['local_count_changes_1000']=local_count_changes_1000


In [ ]:
#get shap values for the SNPs with the highest delta 

#create the explainers 
model_wrapper=(model.input, model.layers[-1].output)
count_explainer=shap.DeepExplainer(model_wrapper,data=create_background,combine_mult_and_diffref=combine_mult_and_diffref_1d)
prof_explainer = create_explainer(model,ischip=False)

In [ ]:
count_preds=count_preds_cases
profile_preds=profile_preds_cases
snp_to_seq=snp_to_seq_cases


In [ ]:
not_processed=[]
ids=[]
counts_ref_list=[]
profile_ref_list=[]
for i in cluster_case_motif1[['details']].drop_duplicates().itertuples():
    #print(i)
    try:    
        a=i[1]
        a1=a.split('_')[0]
        a2=int(a.split('_')[1])
        a2
        #get the signal in probability space 
        from sklearn.preprocessing import normalize
        signal=np.nan_to_num(bw.values(a1,a2-500,a2+500))
        signal=signal/np.sum(signal)
        #print(signal.shape)
        #let's explain chr6_161153273_C_T
        rsid=a
        #print(a)
        #signal=math.log(sum(pbw.values(a1,a2-500,a2+500)))
        cur_seq_ref=np.expand_dims(snp_to_seq[rsid]['ref'],axis=0)
        cur_seq_alt=np.expand_dims(snp_to_seq[rsid]['alt'],axis=0)
        count_pred_ref=count_preds[rsid]['ref']
        count_pred_alt=count_preds[rsid]['alt']
        pred_ref=profile_preds[rsid]['ref']
        pred_alt=profile_preds[rsid]['alt']
        pred_prob_ref=softmax(profile_preds[rsid]['ref'],axis=0)
        pred_prob_alt=softmax(profile_preds[rsid]['alt'],axis=0)
        profile_explanations_ref=prof_explainer(cur_seq_ref,None)
        count_explanations_ref=np.squeeze(count_explainer.shap_values(cur_seq_ref)[0])
        profile_explanations_alt=prof_explainer(cur_seq_alt,None)
        count_explanations_alt=np.squeeze(count_explainer.shap_values(cur_seq_alt)[0])
        ids.append(i[1])
        counts_ref_list.append(count_explanations_ref)
        profile_ref_list.append(profile_explanations_ref)
    except Exception as e:
        print(str(e))
        not_processed.append(i)


In [ ]:
count_preds=count_preds_controls
profile_preds=profile_preds_controls
snp_to_seq=snp_to_seq_controls


In [ ]:
not_processed_control=[]
ids_control=[]
counts_ref_list_control=[]
profile_ref_list_control=[]
for i in cluster_control_motif1[['details']].drop_duplicates().itertuples():
    #print(i)
    try:    
        a=i[1]
        a1=a.split('_')[0]
        a2=int(a.split('_')[1]) 
        from sklearn.preprocessing import normalize
        signal=np.nan_to_num(bw.values(a1,a2-500,a2+500))
        signal=signal/np.sum(signal)
        #print(signal.shape)
        rsid=a
        #print(a)
        cur_seq_ref=np.expand_dims(snp_to_seq[rsid]['ref'],axis=0)
        cur_seq_alt=np.expand_dims(snp_to_seq[rsid]['alt'],axis=0)
        count_pred_ref=count_preds[rsid]['ref']
        count_pred_alt=count_preds[rsid]['alt']
        pred_ref=profile_preds[rsid]['ref']
        pred_alt=profile_preds[rsid]['alt']
        pred_prob_ref=softmax(profile_preds[rsid]['ref'],axis=0)
        pred_prob_alt=softmax(profile_preds[rsid]['alt'],axis=0)
        profile_explanations_ref=prof_explainer(cur_seq_ref,None)
        count_explanations_ref=np.squeeze(count_explainer.shap_values(cur_seq_ref)[0])
        profile_explanations_alt=prof_explainer(cur_seq_alt,None)
        count_explanations_alt=np.squeeze(count_explainer.shap_values(cur_seq_alt)[0])
        ids_control.append(i[1])
        counts_ref_list_control.append(count_explanations_ref)
        profile_ref_list_control.append(profile_explanations_ref)
    except Exception as e:
        print(str(e))
        not_processed_control.append(i)


In [ ]:
importance_Scores=pd.DataFrame()
importance_Scores['id']=ids
importance_Scores['counts_importance']=counts_ref_list
importance_Scores['profile_importance']=profile_ref_list




In [ ]:
importance_Scores_controls=pd.DataFrame()
importance_Scores_controls['id']=ids_control
importance_Scores_controls['counts_importance']=counts_ref_list_control
importance_Scores_controls['profile_importance']=profile_ref_list_control




In [ ]:
chosen_tf=[]
chosen_motif_family=[]
mutations_rsid=[]
for i in cluster_case_motif1[['details']].drop_duplicates().itertuples():
    case_motif1_rows=cluster_case_motif1[cluster_case_motif1['details']==i[1]]
    scores=[]
    tfs=[]
    motif_family=[]
    for j in case_motif1_rows[[4,5,9,6]].itertuples():
        pos_snp=int(i[1].split('_')[1])
        motif_pos1=int(j[1])
        motif_pos2=int(j[2])
        start_pos=pos_snp-motif_pos1
        end_pos=motif_pos2-pos_snp
        start_pos_arr=1057-start_pos
        end_pos_arr=1057+end_pos
        count_imp=importance_Scores[importance_Scores['id']==i[1]]['counts_importance'].values
        #print(count_imp)
        scores.append(np.sum(count_imp[0][start_pos_arr:end_pos_arr])/(end_pos_arr-start_pos_arr))
        tfs.append(j[3])
        motif_family.append(j[4])
        temp=pd.DataFrame()
        temp['scores']=scores
        temp['tfs']=tfs
        temp['motif']=motif_family
    #print(temp[temp['scores']==max(scores)])
    mutations_rsid.append(i)
    chosen_tf.append(temp[temp['scores']==max(scores)]['tfs'].values[0])
    chosen_motif_family.append(temp[temp['scores']==max(scores)]['motif'].values[0])
    #break
        


In [ ]:
#case_motif1=case_motif[~case_motif[10].str.contains('MOUSE')]
case_motif_c0=pd.DataFrame()
case_motif_c0['rsid']=mutations_rsid
case_motif_c0['tf']=chosen_tf
case_motif_c0['motif']=chosen_motif_family



In [ ]:
chosen_tf_control=[]
chosen_motif_family_control=[]
mutations_rsid_control=[]
for i in cluster_control_motif1[['details']].drop_duplicates().itertuples():
    control_motif1_rows=cluster_control_motif1[cluster_control_motif1['details']==i[1]]
    scores=[]
    tfs=[]
    motif_family=[]
    for j in control_motif1[[4,5,9,6]].itertuples():
        pos_snp=int(i[1].split('_')[1])
        motif_pos1=int(j[1])
        motif_pos2=int(j[2])
        start_pos=pos_snp-motif_pos1
        end_pos=motif_pos2-pos_snp
        start_pos_arr=1057-start_pos
        end_pos_arr=1057+end_pos
        count_imp=importance_Scores_controls[importance_Scores_controls['id']==i[1]]['counts_importance'].values
        #print(count_imp)
        scores.append(np.sum(count_imp[0][start_pos_arr:end_pos_arr])/(end_pos_arr-start_pos_arr))
        tfs.append(j[3])
        motif_family.append(j[4])
        temp=pd.DataFrame()
        temp['scores']=scores
        temp['tfs']=tfs
        temp['motif']=motif_family
    #print(temp[temp['scores']==max(scores)])
    mutations_rsid_control.append(i)
    chosen_tf_control.append(temp[temp['scores']==max(scores)]['tfs'].values[0])
    chosen_motif_family_control.append(temp[temp['scores']==max(scores)]['motif'].values[0])
    #break
        


In [ ]:
control_motif_c0=pd.DataFrame()
control_motif_c0['rsid']=mutations_rsid_control
control_motif_c0['tf']=chosen_tf_control
control_motif_c0['motif']=chosen_motif_family_control



In [ ]:
case_list=[]#comment after first cluster run
control_list=[]#comment after first cluster run
case_list.append(case_motif_c0)
control_list.append(control_motif_c0)


In [ ]:
####repeat from the start of the motif block till here for each of the 

In [ ]:
case_mut=pd.DataFrame()
for i in case_list:
    case_mut=case_mut.append(i)

In [ ]:
print(case_mut.tail())
print(case_mut.shape)

In [ ]:
control_mut=pd.DataFrame()
for i in control_list:
    control_mut=control_mut.append(i)

In [ ]:
case_mut['Mutation type']='Case'
control_mut['Mutation type']='Control'

In [ ]:
allmut_motif=case_mut.append(control_mut)

allmut_motif['Chromosome']=allmut_motif['rsid'].apply(lambda x : str(x).replace('(','').replace(')','').split(',')[1].split('_')[0].replace("details='",""))
allmut_motif['Position']=allmut_motif['rsid'].apply(lambda x : str(x).replace('(','').replace(')','').split(',')[1].split('_')[1])
allmut_motif['Reference']=allmut_motif['rsid'].apply(lambda x : str(x).replace('(','').replace(')','').split(',')[1].split('_')[2])
allmut_motif['Alternate']=allmut_motif['rsid'].apply(lambda x : str(x).replace('(','').replace(')','').split(',')[1].split('_')[3].replace("'",""))
del allmut_motif['rsid']
allmut_motif[0:5]

allmut_motif.columns=['TF Name','Motif Family','Mutation Type','Chromosome','Position','Reference','Alternate']
